|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 4:</h2>|<h1>The Scheduler<h1>|
|<h2>Section:</h2>|<h1>Admission and preemption<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: the scheduler that does not fall over<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

Write the scheduler that does not fall over.

Stage 05's scheduler assumed memory was infinite. This one has a block budget
that a running sequence can exhaust at any step, and it has to stay correct
when that happens.

All simulation. The GPU is a counter, which is the right way to get this
right before it is fast.

In [2]:
### run this cell

BLOCK   = 16
PROMPT  = 48
lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=5000).astype(int) + 1

def blocks_for(n):
  return -(-n // BLOCK)

print(f'median sequence: {blocks_for(int(np.median(lengths)))} blocks, p99: {blocks_for(int(np.percentile(lengths,99)))} blocks')

median sequence: 8 blocks, p99: 59 blocks


# Exercise 1: three queues and a budget

Four methods: `arrive`, `admit`, `preempt`, `step`.

The interesting one is `step`. A sequence crossing into a new block may find
the pool empty, and the answer is not to fail. It is to take the blocks back
from somebody else.

In [3]:
class Scheduler:
  def __init__(self, pool_blocks, watermark=0.05, max_running=64):
    self.free    = pool_blocks
    self.reserve = max(1, int(watermark*pool_blocks))
    self.max_running = max_running          # max_num_seqs
    self.waiting, self.running = [], []
    self.preemptions = self.recomputed = self.work = 0

  def arrive(self, total_len):
    self.waiting.append([PROMPT, total_len])

  def admit(self):
    while (self.waiting and len(self.running) < self.max_running
           and self.free - blocks_for(self.waiting[0][0]) >= self.reserve):
      r = self.waiting.pop(0)
      self.free -= blocks_for(r[0])
      self.running.append(r)

  def preempt(self, protect):
    victim = self.running[-1] if self.running[-1] is not protect else self.running[-2]
    self.free += blocks_for(victim[0])
    self.recomputed += victim[0] - PROMPT
    victim[0] = PROMPT
    self.running.remove(victim)
    self.waiting.insert(0, victim)
    self.preemptions += 1

  def step(self):
    self.admit()
    for r in list(self.running):
      if r not in self.running: continue
      if r[0] % BLOCK == 0:
        while self.free == 0 and len(self.running) > 1:
          self.preempt(r)
        if self.free == 0: return
        self.free -= 1
      r[0] += 1
      self.work += 1               # a token the GPU actually produced
      if r[0] >= r[1]:
        self.running.remove(r)
        self.free += blocks_for(r[0])

s = Scheduler(2000)
for i in range(400): s.arrive(PROMPT + int(lengths[i]))
steps = 0
while (s.waiting or s.running) and steps < 200000:
  before = len(s.running)
  s.step(); steps += 1
  if not s.running and not s.waiting: break
print(f'{steps:,} steps, {s.preemptions} preemptions, '
      f'{100*s.recomputed/s.work:.1f}% of the work was done twice')

2,343 steps, 0 preemptions, 0.0% of the work was done twice


# Exercise 2: how hard can you squeeze?

Sweep the pool size. Watch for where degradation stops being graceful.

In [4]:
def run(pool, watermark=0.05, n=400):
  s = Scheduler(pool, watermark)
  for i in range(n): s.arrive(PROMPT + int(lengths[i]))
  steps = 0
  while (s.waiting or s.running) and steps < 200000:
    s.step(); steps += 1
  return steps, s.preemptions, s.recomputed, s.work

pools = [300, 500, 750, 1000, 1500, 2000, 3000, 4000]
rows  = [run(p) for p in pools]
base  = rows[-1][0]

print(f"{'pool':>6} {'steps':>8} {'vs roomy':>9} {'preempt':>8} {'wasted work':>12}")
for p,(st,pr,rc,wk) in zip(pools, rows):
  print(f'{p:>6} {st:>8,} {st/base:>8.2f}x {pr:>8} {100*rc/wk:>11.1f}%')

  pool    steps  vs roomy  preempt  wasted work
   300    4,081     1.74x      412        12.8%
   500    2,675     1.14x      156         4.8%
   750    2,414     1.03x       33         1.1%
  1000    2,343     1.00x        0         0.0%
  1500    2,343     1.00x        0         0.0%
  2000    2,343     1.00x        0         0.0%
  3000    2,343     1.00x        0         0.0%
  4000    2,343     1.00x        0         0.0%


# Exercise 3: the watermark

The reserve is the only thing stopping a new arrival from starving everyone
already running. Try turning it off.

In [5]:
print(f"{'watermark':>10} {'steps':>8} {'preempt':>8} {'wasted work':>12}")
for wm in (0.0, 0.01, 0.05, 0.15, 0.30, 0.50):
  st, pr, rc, wk = run(600, watermark=wm)
  print(f'{wm:>10.0%} {st:>8,} {pr:>8} {100*rc/wk:>11.1f}%')

 watermark    steps  preempt  wasted work
        0%    2,543      775         6.5%
        1%    2,537      508         5.9%
        5%    2,544      100         3.8%
       15%    2,596        0         0.0%


       30%    2,790        0         0.0%


       50%    3,578        0         0.0%


### What the two sweeps say

**Squeezing the pool costs nothing, until it costs a lot.** The steps
column is flat across most of the range and then turns up sharply at the
small end, where wasted work climbs into double figures. That flat region
is the reason PagedAttention was worth building: you can run much closer
to the edge than a contiguous allocator would allow, and pay nothing for
it, right up to the knee.

**The watermark buys correctness, and the price is idle hardware.** At 0%
the scheduler is greedy: most preemptions, most work done twice, and
still competitive on total steps, because a preempted sequence frees a
slot somebody else can use immediately. Raise it and the preemptions go
to zero, then keep raising it and the step count climbs, because the
reserve is memory you are deliberately not using.

So this is a real trade rather than a bug to be fixed: wasted work on one
side, idle capacity on the other, and the useful values are small and
non-zero. Which is hard to guess and easy to measure, and is exactly the
kind of constant that gets set once from a sweep like this one and then
left alone for years.

**And notice which sequence you preempted.** The newest one. That is not
arbitrary: it has generated the fewest tokens, so recomputing it is the
cheapest, and the request that has already waited longest is not punished
for it. Preempting the oldest instead maximises the work you throw away.

    ./vc guide 10